# IBKR API notebook

#### Connection

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from ib_async import *
import pandas as pd
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

2025-04-22 15:06:29,664 - INFO - Connecting to 127.0.0.1:7497 with clientId 14...
2025-04-22 15:06:29,665 - INFO - Connected
2025-04-22 15:06:29,667 - INFO - Logged on to server version 178
2025-04-22 15:06:29,709 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm.nj
2025-04-22 15:06:29,709 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfuture
2025-04-22 15:06:29,710 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarm
2025-04-22 15:06:29,710 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:cashfarm
2025-04-22 15:06:29,710 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm
2025-04-22 15:06:29,710 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:euhmds
2025-04-22 15:06:29,710 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:cashhmds
2025-04-22 15:06

✅ Connected to IBKR API


2025-04-22 16:55:31,431 - ERROR - Peer closed connection.


## Request Historical data

#### Choose your contract

In [5]:
#contract = CFD('IBUST100', 'SMART', 'USD')
contract = Forex(pair="EURUSD", exchange='IDEALPRO')
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
    print()

Contract Detail 1:
  secType: CASH
  conId: 12087792
  symbol: EUR
  exchange: IDEALPRO
  longName: European Monetary Union Euro
  timezoneId: US/Eastern
  tradingHours:   20250421:1715-20250422:1700
  20250422:1715-20250423:1700
  20250423:1715-20250424:1700
  20250424:1715-20250425:1700
  20250426:CLOSED
  20250427:1715-20250428:1700
  liquidHours:   20250421:1715-20250422:1700
  20250422:1715-20250423:1700
  20250423:1715-20250424:1700
  20250424:1715-20250425:1700
  20250426:CLOSED
  20250427:1715-20250428:1700
  minSize: 0.01



#### Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")

### Request historical data function

**End date choice**

In [ ]:
save_path = "./database/AAPL_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
logging.info(f"First date in the DataFrame: {first_date}")
logging.info(f"End date: {end_date}")

In [6]:
#today's date
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [ ]:
historical_data_interval = '10 secs' 
request_duration = '5 Y'
price_source = 'MIDPOINT'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

Convert the list of bars to a data frame and print the first and last rows:

In [8]:
bars[0]
df = util.df(bars)
print("DataFrame shape:", df.shape)

display(df.head(n=20))
display(df.tail(n=20))

DataFrame shape: (1319803, 8)


,date,open,high,low,close,volume,average,barCount
0,2024-09-15 21:15:00+00:00,1.107770,1.107805,1.107770,1.107805,-1.0,-1.0,-1
1,2024-09-15 21:15:10+00:00,1.107805,1.107840,1.107805,1.107810,-1.0,-1.0,-1
2,2024-09-15 21:15:20+00:00,1.107810,1.107810,1.107800,1.107800,-1.0,-1.0,-1
3,2024-09-15 21:15:30+00:00,1.107800,1.107835,1.107800,1.107835,-1.0,-1.0,-1
4,2024-09-15 21:15:40+00:00,1.107835,1.107835,1.107835,1.107835,-1.0,-1.0,-1
5,2024-09-15 21:15:50+00:00,1.107835,1.107835,1.107795,1.107815,-1.0,-1.0,-1
6,2024-09-15 21:16:00+00:00,1.107815,1.107835,1.107810,1.107835,-1.0,-1.0,-1
7,2024-09-15 21:16:10+00:00,1.107835,1.107835,1.107810,1.107815,-1.0,-1.0,-1
8,2024-09-15 21:16:20+00:00,1.107815,1.107840,1.107810,1.107835,-1.0,-1.0,-1
9,2024-09-15 21:16:30+00:00,1.107835,1.107840,1.107825,1.107840,-1.0,-1.0,-1


,date,open,high,low,close,volume,average,barCount
1319783,2025-04-22 13:03:50+00:00,1.146605,1.146645,1.146550,1.146605,-1.0,-1.0,-1
1319784,2025-04-22 13:04:00+00:00,1.146605,1.146755,1.146605,1.146665,-1.0,-1.0,-1
1319785,2025-04-22 13:04:10+00:00,1.146665,1.146715,1.146655,1.146710,-1.0,-1.0,-1
1319786,2025-04-22 13:04:20+00:00,1.146710,1.146860,1.146695,1.146815,-1.0,-1.0,-1
1319787,2025-04-22 13:04:30+00:00,1.146815,1.146900,1.146705,1.146755,-1.0,-1.0,-1
1319788,2025-04-22 13:04:40+00:00,1.146755,1.146770,1.146510,1.146515,-1.0,-1.0,-1
1319789,2025-04-22 13:04:50+00:00,1.146515,1.146700,1.146515,1.146645,-1.0,-1.0,-1
1319790,2025-04-22 13:05:00+00:00,1.146645,1.146810,1.146645,1.146750,-1.0,-1.0,-1
1319791,2025-04-22 13:05:10+00:00,1.146750,1.146755,1.146555,1.146555,-1.0,-1.0,-1
1319792,2025-04-22 13:05:20+00:00,1.146555,1.146660,1.146550,1.146640,-1.0,-1.0,-1


Save your pulled data in a dataframe

Compression possibilities sorted by compression ratio from the lowest to the highest : 
- `snappy`

- `gzip`

- `brotli`

#### Checking if volume and average columns are empty or not and remove it if empty

In [ ]:
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
df = df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
display(df.head(n=30))


Construction du nom du fichier et sauvegarde en `.parquet` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate.parquet`

In [ ]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol
price_source = "ASK"
# Construire le nom du fichier
start_date = df['date'].iloc[0].strftime('%Y%m%d')
end_date = df['date'].iloc[-1].strftime('%Y%m%d')
# Structure
save_path = f"../marketData/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}_{price_source}.parquet"

# Sauvegarder le DataFrame en fichier parquet
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

In [ ]:
import pandas as pd
save_path = "../marketData/NDX_10secs_20220214_to_20250411.parquet"
#save_path = "../marketData/NDX_10secs_20220131_to_20250403.parquet"

# Load the parquet file into a DataFrame

retrieved_df = pd.read_parquet(save_path)
retrieved_df['date'] = retrieved_df['date'].dt.tz_convert('Europe/Paris')
print(type(retrieved_df['date'].iloc[0]))

# Display the first and last rows of the DataFrame
display(retrieved_df.head())
display(retrieved_df.tail())

Instruct the notebook to draw plot graphics inline:

In [ ]:
%matplotlib inline

Plot the close data

In [ ]:
df.plot(y='close');

There is also a utility function to plot bars as a candlestick plot. It can accept either a DataFrame or a list of bars. Here it will print the last 100 bars:

In [ ]:
util.barplot(bars[-100:], title=contract.symbol);

## Historical data with realtime updates

A new feature of the API is to get live updates for historical bars. This is done by setting `endDateTime` to an empty string and the `keepUpToDate` parameter to `True`.

Let's get some bars with an keepUpToDate subscription:

In [ ]:
bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='900 S',
        barSizeSetting='10 secs',
        whatToShow='MIDPOINT',
        useRTH=True,
        formatDate=1,
        keepUpToDate=True)

Replot for every change of the last bar:

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

def onBarUpdate(bars, hasNewBar):
    plt.close()
    plot = util.barplot(bars)
    clear_output(wait=True)
    display(plot)

bars.updateEvent += onBarUpdate

ib.sleep(10)
ib.cancelHistoricalData(bars)

Realtime bars
------------------

With ``reqRealTimeBars`` a subscription is started that sends a new bar every 5 seconds.

First we'll set up a event handler for bar updates:

In [ ]:
def onBarUpdate(bars, hasNewBar):
    print(bars[-1])

Then do the real request and connect the event handler,

In [ ]:
bars = ib.reqRealTimeBars(contract, 5, 'MIDPOINT', False)
bars.updateEvent += onBarUpdate

let it run for half a minute and then cancel the realtime bars.

In [ ]:
ib.sleep(30)
ib.cancelRealTimeBars(bars)

The advantage of reqRealTimeBars is that it behaves more robust when the connection to the IB server farms is interrupted. After the connection is restored, the bars from during the network outage will be backfilled and the live bars will resume.

reqHistoricalData + keepUpToDate will, at the moment of writing, leave the whole API inoperable after a network interruption.

### Request historical market news

In [ ]:
news_providers = ib.reqNewsProviders()
print("News Providers:", news_providers)
for provider in news_providers:
    print(f"Code: {provider.code}, Name: {provider.name}")

In [ ]:
article = ib.reqNewsArticle(providerCode="BRFG")
print(article)

In [ ]:
historical_news = ib.reqHistoricalNews(
    conId=contract.conId,  # ID du contrat
    providerCodes="BRFG",
    startDateTime="20250401 00:00:00",
    endDateTime="20250405 23:59:59",
    totalResults=10
)
for news in historical_news:
    print(f"Date: {news.time}, Title: {news.headline}")

In [ ]:
newsbulletin = ib.reqNewsBulletins(allMessages=True)

In [ ]:
ib.disconnect()

## Additional features

#### Data pre processing

In [ ]:
import os
import pandas as pd
from igtrader.Strategies.Helpers import load_data, resample_ohlc

In [ ]:
symbol = 'NDX'
interval = '10secs'
start_date = '20220214'
end_date = '20250411'

save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}.parquet"
df = pd.read_parquet(save_path, engine='pyarrow')
df

In [ ]:
interval= 'secs'
df = resample_ohlc(df, interval)
df

In [ ]:
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}.parquet"
df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")

In [ ]:
# Remove the 'barCount' column from the DataFrame
df = df.drop(columns=['barCount'])

# Display the updated DataFrame
display(df.head())
df.to_parquet(save_path, index=True, compression=None)